# FINS5557 project notebook

Our one notebook for exploring the data and developing the model. Code that works and gets reused (loading, features, training, SHAP, fairness, PSI) is moved into `src/`, and this notebook imports it, so the app and the notebook always use the same steps.

| Section | Status |
|---|---|
| 0. Setup | done |
| 1. Data: exploration and checks | done |
| 2. Features | done |
| 3. Models: baseline vs random forest vs XGBoost | done |
| 4. Calibration and threshold | done |
| 5. Explainability (SHAP) and fairness | to do |

See `docs/BUILD_PLAN.md` for the full plan.

**Before you run:** activate `.venv`, pick it as this notebook's kernel (top right, **Select Kernel**), then run top to bottom. Clear outputs before committing.

## 0. Setup
### 0.1 Install packages
Installs `requirements-dev.txt` (which includes `requirements.txt`). Skip if you've already installed them.

In [ ]:
%pip install -q -r ../requirements-dev.txt

### 0.2 Imports and package versions
Prints each version, so a broken setup shows up here rather than half-way through.

`autoreload` picks up changes to `src/` files. If you still get an import error after code changes, use **Restart** in the notebook toolbar, then **Run All**.

In [ ]:
# Reload src/ modules automatically when their files change, so edits to src/
# show up without restarting the kernel.
%load_ext autoreload
%autoreload 2

import sys
import warnings
from importlib.metadata import version
from pathlib import Path

# Make `src` importable whether the notebook runs from the project root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.config import GOOGLE_API_KEY, LLM_BACKEND, PROTECTED, RANDOM_STATE, TARGET
from src.data import age_years, clean, features_target, load_raw, split

warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.3f}".format)

print("Python", sys.version.split()[0])
for pkg in ["pandas", "numpy", "scikit-learn", "xgboost", "shap", "streamlit", "python-dotenv", "joblib"]:
    print(f"{pkg:15s} {version(pkg)}")

### 0.3 Settings check
Reports only whether an API key was found, never the key itself.

In [ ]:
print("Random seed:", RANDOM_STATE)
print("LLM backend:", LLM_BACKEND)
print("GOOGLE_API_KEY found:", bool(GOOGLE_API_KEY))

## 1. Data: exploration and checks

**Data:** Home Credit Default Risk (Kaggle, 2018), `application_train.csv`: 307,511 consumer loan applications, one row each, with what the lender knew at the time of application. `TARGET` (renamed `default`) = 1 if the client had payment difficulties on the loan. Source: https://www.kaggle.com/competitions/home-credit-default-risk/data (competition rules: not redistributed, so `data/raw/` is gitignored).

We load only the columns we use (inputs listed in `src/config.py`, plus the ID, target and the protected attributes for the fairness audit).

**Findings** (fill in after running; this becomes the Data section of the report):
- Default rate: **8.1%** (24,825 of 307,511 applications had payment difficulties). Classes are imbalanced about 11:1, so accuracy is meaningless (predicting "no default" for everyone scores 91.9%); we use AUC, Brier score and profit instead. The stratified 60/20/20 split keeps 8.1% in train (184,506 rows), validation (61,502) and test (61,503).
- Strongest early signals: the three **external credit scores**. Moving from the lowest to the highest fifth of `EXT_SOURCE_3` cuts the default rate from 16.4% to 3.4% (`EXT_SOURCE_2`: 15.2% to 3.6%; `EXT_SOURCE_1`: 14.6% to 3.0%). Among categories, lower secondary education (10.9%), renting or living with parents (11.7-12.3%) and "Working" income type (9.6%) are riskier; pensioners (5.4%) and revolving loans (5.5%) are safer.
- Missing data and what it means (thin files): `EXT_SOURCE_1` is missing for 56.4% of applicants and `EXT_SOURCE_3` for 19.8%; `OWN_CAR_AGE` is missing for 66% (no car), `OCCUPATION_TYPE` for 31%. A missing `EXT_SOURCE_3` comes with a higher default rate (9.3% vs 8.1% overall), consistent with a thin credit file, so gaps are kept as information (median fill plus a "was missing" flag).
- Data issues and how we handle them: `DAYS_EMPLOYED = 365243` for 18.0% of rows is Home Credit's "not employed" code (in practice these are all pensioners), turned into a flag. `CODE_GENDER` has 4 rows coded `XNA`, set to missing (audit only). Income has extreme outliers (max 117m; 0.08% above 1m), which the tree models are robust to. No duplicate application IDs.
- Fairness preview: default rates differ by protected group: men 10.1% vs women 7.0%; age falls steadily from 11.5% (20s) to 4.9% (60s); single / civil marriage 9.8-9.9% vs married 7.6%. Gender, age and family status are never model inputs, but other inputs (income type, occupation, education) can act as proxies, so the Step 10 audit checks approval rates by these groups. Education (an input) ranges from 1.8% (academic degree, only 164 people) to 10.9% (lower secondary).

### 1.1 Load the data
Reads `data/raw/application_train.csv`. If the file is missing, the error says where to download it.

In [ ]:
raw = load_raw()
print(raw.shape)  # expect (307511, 27): ID, target, 23 inputs... see config for the exact list
raw.head()

### 1.2 Target: how many applicants default?
About 8% is expected. That is a realistic consumer-lending rate (the Taiwan data had 22%). Predicting "no default" for everyone would score about 92% accuracy, which is why we judge models on AUC, KS and Brier score and use class weights.

In [ ]:
raw[TARGET].value_counts(normalize=True).rename("share")

### 1.3 Data quality checks
Home Credit has many missing values. Missing doesn't always mean bad data: no external credit score means the applicant has little credit history (a thin file).

In [ ]:
missing = raw.isna().mean().sort_values(ascending=False)
print("Share missing per column (columns with any gaps):")
print(missing[missing > 0].round(3).to_string())
print()
print("Duplicate application IDs:", int(raw["SK_ID_CURR"].duplicated().sum()))
print("DAYS_EMPLOYED = 365243 ('not employed' code):", f"{(raw['DAYS_EMPLOYED'] == 365243).mean():.1%}")
print("CODE_GENDER values:", raw["CODE_GENDER"].value_counts().to_dict())

In [ ]:
df = clean(raw)
print("After cleaning, CODE_GENDER:", df["CODE_GENDER"].value_counts(dropna=False).to_dict())

### 1.4 External scores vs default
`EXT_SOURCE_1/2/3` are normalised scores from external data sources (credit bureaus). Expect them to be the strongest predictors, and expect applicants with **no** score to default more.

In [ ]:
for col in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]:
    by_quintile = df.groupby(pd.qcut(df[col], 5), observed=True)[TARGET].mean()
    print(col, "| missing:", f"{df[col].isna().mean():.1%}",
          "| default rate if missing:", f"{df.loc[df[col].isna(), TARGET].mean():.1%}")
    print(by_quintile.round(3).to_string(), "\n")

### 1.5 Amounts and affordability
Income and loan amounts are very skewed (a few huge incomes), which matters for logistic regression (trees don't mind). The repayment-to-income ratio is the classic affordability measure.

In [ ]:
df[["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"]].describe().T

In [ ]:
affordability = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
df.groupby(pd.qcut(affordability, 5), observed=True)[TARGET].mean().rename("default_rate by repayment/income")

In [ ]:
# Categorical inputs: default rate by category (largest groups first)
for col in ["NAME_CONTRACT_TYPE", "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "NAME_HOUSING_TYPE"]:
    print(df.groupby(col, dropna=False)[TARGET].agg(applicants="size", default_rate="mean")
            .sort_values("applicants", ascending=False).round(3), "\n")

### 1.6 Fairness preview
Gender, age and family status are **not** model inputs, but we audit the model's decisions by them later. Here are the raw default rates by group as a baseline.

In [ ]:
age_band = pd.cut(age_years(df), [20, 30, 40, 50, 60, 70], right=False)
for name, group in [("Gender", df["CODE_GENDER"]), ("Age band", age_band),
                    ("Family status", df["NAME_FAMILY_STATUS"]), ("Education", df["NAME_EDUCATION_TYPE"])]:
    print(name)
    print(df.groupby(group, observed=True, dropna=False)[TARGET].agg(applicants="size", default_rate="mean").round(3), "\n")

### 1.7 Train / validation / test split
Stratified 60/20/20, done before any model fitting. The default rate should be the same in all three parts.

In [ ]:
train, val, test = split(df)
pd.DataFrame({
    "rows": [len(train), len(val), len(test)],
    "default_rate": [train[TARGET].mean(), val[TARGET].mean(), test[TARGET].mean()],
}, index=["train", "validation", "test"])

In [ ]:
X_train, y_train = features_target(train)
print(X_train.shape[1], "model inputs; protected attributes removed:", PROTECTED)
list(X_train.columns)

## 2. Features

New features built by `add_features()` in `src/features.py`, each a standard credit-risk idea:

| Feature | Definition | Credit-risk meaning |
|---|---|---|
| `credit_income_ratio` | loan / annual income | Leverage: how big the loan is for this income |
| `annuity_income_ratio` | repayment / income | Affordability (debt-service ratio), what responsible-lending checks look at |
| `credit_term` | loan / repayment | Roughly the number of repayments; longer loans are riskier |
| `goods_credit_ratio` | goods price / loan | Below 1 = borrowing extra cash on top of the goods |
| `years_employed`, `not_employed` | from `DAYS_EMPLOYED`; 365243 = not employed | Job stability; the code must not be read as 1,000 years in a job |
| `ext_source_mean` | mean of the available external scores | Overall external credit view |
| `ext_source_missing` | number of missing external scores (0-3) | Thin-file signal |
| `income_per_person` | income / family members | Income spread over the household |

Divisions are safe: a zero, negative or missing denominator gives a missing value, which the pipeline fills with the training median and flags with a "was missing" column. Everything here uses the **training split only**.

**Findings** (fill in after running):
- Strongest engineered feature: **`ext_source_mean`** (correlation with default -0.225; default rate 18.8% in the lowest fifth vs 2.5% in the highest), far ahead of the rest. Next are `years_employed` (-0.077; 11.3% for under 1.7 years in the job vs 5.2% for over 10 years) and `goods_credit_ratio` (-0.064; 11.7% when the loan is well above the goods price, i.e. extra cash borrowed, vs 4.2% when the goods cost more than the loan).
- Shape of the affordability relationship: **weak and flat at the top**. Default rises only from 7.2% (repayment under 10% of income) to 8.7-8.8% around 19-25%, then levels off (8.4% above 25%). `credit_income_ratio` is hump-shaped (7.3% → 8.8% → 7.2%). Likely reason: the lender already limited loan size by income when approving, so stated affordability adds little on its own. `credit_term` is also non-monotonic: loans of 14-22 repayments default most (9.7-9.9%) and the longest loans least (6.1-6.6%), against the textbook expectation, probably because longer terms were granted to stronger applicants. The tree models can use these non-linear shapes; logistic regression cannot.
- Thin-file effect (`ext_source_missing`): default rises from 7.4% with all three external scores to 8.1% with one missing and 10.0% with two missing (the 109 applicants with none are too few to read). Separately, `not_employed` applicants default *less* (5.5% vs 8.6%) because they are pensioners, so this flag mostly captures age; worth noting for the fairness audit.

In [ ]:
from src.config import ENGINEERED_FEATURES
from src.features import add_features, build_preprocessor

train_f = add_features(train)
train_f[ENGINEERED_FEATURES].describe().T

### 2.1 Each feature against the default rate
Correlation gives a quick ranking; the quintile tables show the shape of each relationship (a correlation near 0 can still hide a U-shape).

In [ ]:
train_f[[*ENGINEERED_FEATURES, TARGET]].corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).rename("correlation")

In [ ]:
for col in ["annuity_income_ratio", "credit_income_ratio", "credit_term", "goods_credit_ratio", "years_employed", "ext_source_mean"]:
    print(train_f.groupby(pd.qcut(train_f[col], 5, duplicates="drop"), observed=True)[TARGET].mean().round(3).rename(col), "\n")
print(train_f.groupby("ext_source_missing")[TARGET].agg(applicants="size", default_rate="mean").round(3))
print(train_f.groupby("not_employed")[TARGET].agg(applicants="size", default_rate="mean").round(3))

### 2.2 Preprocessing pipeline
`build_preprocessor()`: numeric columns get median imputation plus a 0/1 "was missing" column, then scaling (needed for logistic regression, harmless for trees). Categorical columns get "Unknown" for missing, rare categories (< 1% of training rows) grouped, then one-hot encoding. Fitted on the training split only, then applied unchanged to validation. Protected attributes and the ID are dropped. Pattern adapted from Code Bank W06-04.

In [ ]:
pre = build_preprocessor()
X_train_ready = pre.fit_transform(train_f)
X_val_ready = pre.transform(add_features(val))
print("train:", X_train_ready.shape, " validation:", X_val_ready.shape)
list(pre.get_feature_names_out())

## 3. Models: baseline vs random forest vs XGBoost

Training lives in `src/train.py`. Run it once from the project root (about 5 to 10 minutes on the full data):

```bash
python -m src.train
```

It tunes each model with 3-fold stratified cross-validation on the **training split only**, scores all three on the **validation split**, applies the model-choice rule and saves `models/model.joblib`, `models/metadata.json` and `reports/model_comparison.csv`. This section reads those files instead of retraining. The test split is still untouched.

| Model | Why it's in the comparison | Imbalance handling |
|---|---|---|
| Logistic regression | Traditional scorecard baseline (Code Bank W01-06, W05-02): linear, transparent, easy to defend to a regulator | `class_weight="balanced"` |
| Random forest | Many decision trees averaged: captures non-linear patterns and interactions | `class_weight="balanced"` |
| XGBoost | Boosted trees, usually the strongest on tabular credit data (Code Bank W02-06, W05-03) | `scale_pos_weight` = non-defaulters / defaulters ≈ 11.4 |

Every model is one pipeline (feature engineering → scaling / one-hot → model), so features are fitted inside each CV fold and the saved model takes raw applicant columns.

**Model-choice rule** (`MIN_AUC_GAIN` in `src/config.py`): take the highest validation AUC, but a tree model has to beat logistic regression by at least 0.01 AUC to justify the loss of transparency.

**Findings** (fill in after retraining on Home Credit):
- Model comparison:
- Chosen model and why:
- Overfitting check (CV vs validation AUC):

In [ ]:
import json

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

from src.config import METADATA_PATH, REPORTS_DIR

comparison = pd.read_csv(REPORTS_DIR / "model_comparison.csv", index_col="model")
metadata = json.loads(METADATA_PATH.read_text())
print("Chosen:", metadata["chosen_model"], "|", metadata["reason"])
print("Best settings:", metadata["best_params"])
comparison

### 3.1 ROC curves on the validation split
Each curve shows the trade-off between catching defaulters (true positive rate) and wrongly flagging good customers (false positive rate) across all cut-offs. A higher curve means better ranking; the dashed line is a coin flip (AUC 0.5).

In [ ]:
val_preds = pd.read_csv(REPORTS_DIR / "validation_predictions.csv")

fig, ax = plt.subplots(figsize=(6, 5))
for name in comparison.index:
    fpr, tpr, _ = roc_curve(val_preds[TARGET], val_preds[name])
    ax.plot(fpr, tpr, label=f"{name} (AUC {comparison.loc[name, 'auc']:.3f})")
ax.plot([0, 1], [0, 1], "--", color="grey", label="random (AUC 0.500)")
ax.set(xlabel="False positive rate (good customers flagged)",
       ylabel="True positive rate (defaulters caught)", title="ROC curves: validation split")
ax.legend(loc="lower right")
plt.show()

## 4. Calibration and threshold

Code in `src/threshold.py`, run as part of `python -m src.train`. Three steps turn a model score into a credit decision:

1. **Calibrate.** Class weighting makes the chosen model rank customers well but overstate risk (its average score is well above the ~8% default rate). A calibrator fitted on the validation split maps scores to **PDs** that match observed default rates. Sigmoid (Platt) and isotonic calibration are compared on out-of-fold Brier score, so neither is judged on rows it was fitted on.
2. **Choose the cut-off from the lender's economics**, not 0.5. Approving a customer earns `MARGIN × EAD` if they repay and loses `LGD × EAD` if they default, so the expected loss is `PD × LGD × EAD` (the Basel formula). Assumptions in `src/config.py`: LGD = 60%, net margin over the loan's life = 15%, EAD = loan amount. The break-even PD is `margin / (margin + LGD)` = 20%. The cut-off is the PD that maximises **realised** profit on the validation split.
3. **Decide:** approve below `cut-off - 2 points`, decline above `cut-off + 2 points`, refer the band in between to an analyst (`REFER_BAND` in `src/config.py`).

The test split is then scored **once**, at the chosen cut-off.

**Findings** (fill in after retraining on Home Credit):
- Calibration:
- Cut-off vs break-even PD (20%):
- Test results:
- Refer band share:

In [ ]:
from sklearn.calibration import calibration_curve

from src.threshold import break_even_pd, decide

print("Calibration method:", metadata["calibration"]["method"])
print("Validation Brier:", metadata["calibration"]["validation_brier"])
print("Assumptions:", metadata["assumptions"])
print(f"Cut-off: decline at PD >= {metadata['threshold']:.3f}, refer band +/- {metadata['refer_band']}")

### 4.1 Reliability curve
Customers grouped into 10 bins by predicted risk: average prediction (x) against the share that actually defaulted (y). A calibrated model sits on the diagonal. Validation split, out-of-fold calibrated PDs.

In [ ]:
chosen = metadata["chosen_model"]
fig, ax = plt.subplots(figsize=(6, 5))
for label, scores in [(f"{chosen}, uncalibrated", val_preds[chosen]), ("calibrated PD", val_preds["calibrated_pd"])]:
    observed, predicted = calibration_curve(val_preds[TARGET], scores, n_bins=10, strategy="quantile")
    ax.plot(predicted, observed, "o-", label=label)
ax.plot([0, 1], [0, 1], "--", color="grey", label="perfect calibration")
ax.set(xlabel="Predicted probability of default", ylabel="Observed default rate",
       title="Reliability curve: validation split")
ax.legend()
plt.show()

### 4.2 Profit curve
Profit on the validation split if everyone below each cut-off is approved. **Realised** uses what actually happened; **expected** uses the PDs. Too low a cut-off turns away profitable customers, too high a cut-off approves loss-making ones.

In [ ]:
curve = pd.read_csv(REPORTS_DIR / "profit_curve.csv")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(curve["threshold"], curve["realised_profit"] / 1e6, label="realised profit")
ax.plot(curve["threshold"], curve["expected_profit"] / 1e6, label="expected profit (from PDs)")
ax.axvline(metadata["threshold"], color="black", label=f"chosen cut-off ({metadata['threshold']:.3f})")
ax.axvline(break_even_pd(), color="grey", linestyle="--", label=f"break-even PD ({break_even_pd():.3f})")
ax.axvspan(metadata["threshold"] - metadata["refer_band"], metadata["threshold"] + metadata["refer_band"],
           color="orange", alpha=0.15, label="refer band")
ax.set(xlabel="PD cut-off (approve below)", ylabel="Profit, million (loan currency)", title="Profit by cut-off: validation split")
ax.legend()
plt.show()

curve.set_index("threshold").loc[[0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.50]]

### 4.3 How wide should the refer band be?
Share of validation applicants sent to an analyst for different band widths around the cut-off.

In [ ]:
pds = val_preds["calibrated_pd"]
pd.DataFrame({
    band: pd.Series(decide(pds, metadata["threshold"], band)).value_counts(normalize=True)
    for band in [0.01, 0.02, 0.03, 0.05]
}).T.rename_axis("refer band (+/-)").round(3)[["approve", "refer", "decline"]]

### 4.4 Final test-set results (reported once)
The confusion matrix treats "decline" as predicting default (refer band ignored).

In [ ]:
test_metrics = json.loads((REPORTS_DIR / "test_metrics.json").read_text())
cm = test_metrics["confusion_matrix"]
print(pd.DataFrame([[cm["approved_repaid"], cm["declined_repaid"]], [cm["approved_defaulted"], cm["declined_defaulted"]]],
                   index=["actually repaid", "actually defaulted"], columns=["approved", "declined"]))
print()
pd.Series({k: v for k, v in test_metrics.items() if not isinstance(v, dict)}).round(4)

### 4.5 Scoring applicants (Step 5: `src/predict.py`)
This is the interface the app uses. `load_artifacts()` loads the calibrated model and its metadata once (the app caches it with `st.cache_resource`). `predict()` scores one applicant from a dict of raw fields; `predict_batch()` scores a whole table in one vectorised call. Both use the cut-off and refer band saved in `models/metadata.json`, so the app can't drift from what was trained. Protected attributes are not needed: the applicant below is scored without gender, age or family status.

`reasons` and `shap` are empty until the explainability step (section 5) is built.

In [ ]:
from src.predict import load_artifacts, predict, predict_batch

artifacts = load_artifacts()

applicant = features_target(test)[0].iloc[0].to_dict()  # only the fields the app collects
print("Applicant:", applicant)
predict(applicant, artifacts)

In [ ]:
# A missing field gives a clear error instead of a crash deep inside the model
incomplete = {k: v for k, v in applicant.items() if k != "AMT_CREDIT"}
try:
    predict(incomplete, artifacts)
except ValueError as err:
    print("ValueError:", err)

In [ ]:
# Score 1,000 test applicants at once and check the decision mix
scored = predict_batch(test.drop(columns=TARGET).head(1000), artifacts)
print(scored["decision"].value_counts(normalize=True).round(3))
scored[["AMT_CREDIT", "AMT_INCOME_TOTAL", "EXT_SOURCE_2", "pd", "decision"]].head()

## 5. Explainability (SHAP) and fairness
_To do: see `docs/BUILD_PLAN.md`._